# Recurrent Neural Networks (RNN) Fundamentals

## Introduction
Standard Feed-Forward Neural Networks (like Dense networks) assume that all inputs (and outputs) are independent of each other. However, for many tasks, the **sequence** of data matters. 

**Recurrent Neural Networks (RNNs)** are a class of neural networks designed to handle sequential data. They have a "memory" that captures information about what has been calculated so far.

### Common Use Cases
- Time Series Prediction (Stock prices, weather)
- Natural Language Processing (Translation, Sentiment Analysis)
- Speech Recognition

### The RNN Architecture
An RNN processes a sequence of vectors step-by-step. At each step, it takes an input $x_t$ and the previous hidden state $h_{t-1}$ to produce a new hidden state $h_t$.

<center><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/Recurrent_neural_network_unfold.svg/1200px-Recurrent_neural_network_unfold.svg.png" width="600"></center>

*Image Source: Wikimedia Commons*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## 1. Generating Sequential Data
To understand RNNs, let's start with a simple synthetic dataset: a **Sine Wave**. This involves predicting the next value in a sequence based on past values.

In [ ]:
# Generate a sine wave
t = np.arange(0, 100, 0.1)
data = np.sin(t)

# Plot the data
plt.figure(figsize=(10, 6))
plt.plot(t, data)
plt.title("Sine Wave Data")
plt.xlabel("Time")
plt.ylabel("Value")
plt.show()

## 2. Preparing Data for RNNs
RNNs in Keras expect input data in a 3D format: `(batch_size, time_steps, features)`.

- **Batch Size**: Number of samples processed at once.
- **Time Steps**: How many past steps the model looks back to make a prediction.
- **Features**: Dimensionality of the data at each time step (1 for a simple sine wave).

We need to create "windows" of data. For example, if `time_steps=10`, we use values at indices `[0..9]` to predict `[10]`.

In [ ]:
def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(len(data) - time_steps):
        # Inputs: data from i to i + time_steps
        X.append(data[i : i + time_steps])
        # Target: data at i + time_steps (the next value)
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)

TIME_STEPS = 20

X, y = create_sequences(data, TIME_STEPS)

# Reshape X to be [samples, time_steps, features]
# Our feature size is 1 because we only have 1 value (amplitude) at each step
X = X.reshape(X.shape[0], X.shape[1], 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split into Train/Test
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

## 3. Building a Simple RNN Model
We will use the `SimpleRNN` layer. This is the most basic form of RNN, which often suffers from the "vanishing gradient" problem for very long sequences, but is perfect for this simple task.

In [ ]:
model = Sequential()

# SimpleRNN layer
# units=50: Dimensionality of the hidden state
# activation='tanh': Standard for RNNs
# input_shape=(time_steps, features)
model.add(SimpleRNN(units=50, activation='tanh', input_shape=(TIME_STEPS, 1)))

# Output layer
# Dense(1) because we are predicting a single continuous value
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.summary()

## 4. Training

In [ ]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Training History")
plt.show()

## 5. Evaluation & Visualization
Let's see how well our model predicts the sine wave.

In [ ]:
predicted = model.predict(X_test)

plt.figure(figsize=(10, 6))
plt.plot(y_test, label='Actual', color='blue')
plt.plot(predicted, label='Predicted', color='red', linestyle='dashed')
plt.title("RNN Predictions vs Actual Data")
plt.legend()
plt.show()

## Limitations of SimpleRNN
While this worked great for a clean sine wave, `SimpleRNN` struggles with complex, long-term dependencies due to the **Vanishing Gradient Problem**.

In the next notebook, we will introduce **LSTMs (Long Short-Term Memory)** and **GRUs (Gated Recurrent Units)**, which are designed to solve this problem.